# 06 — Parallel Execution & Routing

This notebook demonstrates two orchestration primitives from orqest:

1. **Parallel** — runs multiple agents concurrently via `asyncio`, collects results, and merges them with a custom strategy
2. **Router** — routes input to the right agent based on rule conditions or a classifier function

**Use case (Parallel):** Given a business idea, run 3 analysts in parallel who evaluate it from different angles — market, technical, and risk — then merge the results into a unified investment report.

**Use case (Router):** Route incoming customer support messages to the right specialist — billing, technical, or general support.

**Prerequisites:**
- A `.env` file in the project root (or environment variables) with:
  ```
  LLM_API_KEY=your_api_key
  LLM_MODEL=openai:gpt-4o   # or anthropic:claude-sonnet-4-20250514, google:gemini-2.0-flash, etc.
  ```

## 1. Setup

Load configuration from `.env` — orqest never loads environment variables at import time.

In [1]:
from orqest import load_config

config = load_config()
print(f"Model:    {config.llm_model}")
print(f"API key:  {config.llm_api_key[:8]}...")

Model:    openai:gpt-4.1
API key:  sk-proj-...


---

# Part 1: Parallel — Multi-Perspective Analysis

We want three analysts to evaluate a business idea **simultaneously**:

| Analyst | Focus | Output |
|---------|-------|--------|
| **MarketAnalyst** | Market size, competition, positioning | `MarketAnalysis` |
| **TechAnalyst** | Technical feasibility, stack, timeline | `TechAnalysis` |
| **RiskAnalyst** | Risks, challenges, mitigations | `RiskAnalysis` |

A custom merge function combines all three into a unified `InvestmentReport`.

## 2. Define analyst agents

Each analyst has a focused system prompt and its own Pydantic output type. All three share the same `_run_implementation()` pattern — extract the user message, call the model, return structured output.

In [2]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


# --- Output types ---

class MarketAnalysis(BaseModel):
    """Market analyst's structured output."""
    market_size: str = Field(description="Estimated market size and growth trajectory")
    competitors: list[str] = Field(description="Key competitors in the space")
    positioning: str = Field(description="Recommended market positioning strategy")
    score: int = Field(description="Market attractiveness score from 1-10")


class TechAnalysis(BaseModel):
    """Technical analyst's structured output."""
    feasibility: str = Field(description="Assessment of technical feasibility")
    recommended_stack: list[str] = Field(description="Recommended technology stack components")
    timeline_months: int = Field(description="Estimated months to build an MVP")
    score: int = Field(description="Technical feasibility score from 1-10")


class RiskAnalysis(BaseModel):
    """Risk analyst's structured output."""
    top_risks: list[str] = Field(description="Top 3-5 risks for this venture")
    mitigations: list[str] = Field(description="Suggested mitigation for each risk")
    overall_risk_level: str = Field(description="Low, Medium, or High")
    score: int = Field(description="Risk score from 1-10, where 10 means lowest risk")


# --- Agents ---

class MarketAnalyst(BaseAgent[GlobalState, MarketAnalysis]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="market_analyst",
            system_prompt=(
                "You are a market research analyst. Given a business idea, evaluate "
                "the market size, identify key competitors, and recommend a positioning "
                "strategy. Be specific and data-informed. Provide a market attractiveness "
                "score from 1-10."
            ),
            output_type=MarketAnalysis,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> MarketAnalysis:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


class TechAnalyst(BaseAgent[GlobalState, TechAnalysis]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="tech_analyst",
            system_prompt=(
                "You are a technical feasibility analyst. Given a business idea, assess "
                "whether it can be built with current technology, recommend a tech stack, "
                "and estimate the timeline to MVP. Provide a feasibility score from 1-10."
            ),
            output_type=TechAnalysis,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> TechAnalysis:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


class RiskAnalyst(BaseAgent[GlobalState, RiskAnalysis]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="risk_analyst",
            system_prompt=(
                "You are a risk assessment analyst. Given a business idea, identify the "
                "top 3-5 risks, suggest mitigations for each, and rate the overall risk "
                "level (Low/Medium/High). Provide a risk score from 1-10 where 10 means "
                "lowest risk."
            ),
            output_type=RiskAnalysis,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> RiskAnalysis:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


print("Analyst agents defined: MarketAnalyst, TechAnalyst, RiskAnalyst")

Analyst agents defined: MarketAnalyst, TechAnalyst, RiskAnalyst


## 3. Parallel execution — run all 3, time it

`Parallel` takes a list of steps (agents, functions, or Step objects) and runs them as concurrent `asyncio` tasks. We time the execution to prove all three analysts run simultaneously — the wall-clock time should be close to the **slowest** agent, not the sum of all three.

In [3]:
import time
from orqest.orchestration import Parallel

# Instantiate the analysts
market_analyst = MarketAnalyst(model=config.llm_model, api_key=config.llm_api_key)
tech_analyst = TechAnalyst(model=config.llm_model, api_key=config.llm_api_key)
risk_analyst = RiskAnalyst(model=config.llm_model, api_key=config.llm_api_key)

# Create a Parallel that runs all 3 concurrently
parallel = Parallel(
    steps=[market_analyst, tech_analyst, risk_analyst],
    name="investment_analysis",
)

# The business idea to evaluate
business_idea = (
    "An AI-powered personal finance app that analyzes spending patterns, "
    "negotiates bills on behalf of users, and automatically moves money "
    "between accounts to maximize savings and minimize fees."
)

# Time the parallel execution
t0 = time.monotonic()
result = await parallel.run(business_idea)
parallel_elapsed = time.monotonic() - t0

print(f"Parallel execution completed in {parallel_elapsed:.2f}s")
print(f"Outputs received: {sum(1 for o in result.outputs if o is not None)}/3")
print(f"Errors: {sum(1 for e in result.errors if e is not None)}")

Parallel execution completed in 4.04s
Outputs received: 3/3
Errors: 0


In [4]:
# Inspect the ParallelResult — outputs are ordered by step index
market_result: MarketAnalysis = result.outputs[0]
tech_result: TechAnalysis = result.outputs[1]
risk_result: RiskAnalysis = result.outputs[2]

print("=== Market Analysis ===")
print(f"Market size: {market_result.market_size}")
print(f"Competitors: {', '.join(market_result.competitors)}")
print(f"Positioning: {market_result.positioning}")
print(f"Score: {market_result.score}/10")

print("\n=== Technical Analysis ===")
print(f"Feasibility: {tech_result.feasibility}")
print(f"Stack: {', '.join(tech_result.recommended_stack)}")
print(f"Timeline: {tech_result.timeline_months} months to MVP")
print(f"Score: {tech_result.score}/10")

print("\n=== Risk Analysis ===")
print(f"Risk level: {risk_result.overall_risk_level}")
for risk, mitigation in zip(risk_result.top_risks, risk_result.mitigations):
    print(f"  - {risk}")
    print(f"    Mitigation: {mitigation}")
print(f"Score: {risk_result.score}/10")

=== Market Analysis ===
Market size: The global personal finance software/apps market was valued at approximately $1.3 billion in 2023 and is projected to grow at a CAGR of 16-18% through 2030, reaching around $3 billion. In the US alone, over 70 million adults actively use at least one personal finance app. The intersection of AI-powered features and automated negotiation (a newer trend) is estimated to reach early adoption phase with significant growth expected.
Competitors: Mint (Intuit), YNAB (You Need A Budget), Copilot, Rocket Money (formerly Truebill), PocketGuard, Empower Personal Dashboard (formerly Personal Capital), Albert
Positioning: Position as the smartest, most proactive financial assistant by emphasizing unique AI features: bill negotiation (automation, not just tracking), intelligent money movement, and real, personalized savings strategies. Differentiate through tangible ROI for users, verified savings testimonies, and ease of use. Target tech-savvy millennials, Gen 

## 4. Custom merge — combine into a unified report

By default, `Parallel` uses `MergeStrategy.collect_all` which returns a list of all successful outputs. You can pass any callable as the `merge` parameter to combine results however you like.

Here we define an `InvestmentReport` that synthesizes all three analyses into one document with a composite score.

In [5]:
from typing import Any


class InvestmentReport(BaseModel):
    """Unified report combining all analyst perspectives."""
    business_idea: str
    market: MarketAnalysis | None = None
    tech: TechAnalysis | None = None
    risk: RiskAnalysis | None = None
    composite_score: float = Field(description="Weighted average of all scores")
    recommendation: str = Field(description="Invest, Pass, or Needs More Research")


def merge_into_report(results: list[Any]) -> InvestmentReport:
    """Custom merge function that combines analyst outputs into an InvestmentReport."""
    market = tech = risk = None
    for r in results:
        if isinstance(r, MarketAnalysis):
            market = r
        elif isinstance(r, TechAnalysis):
            tech = r
        elif isinstance(r, RiskAnalysis):
            risk = r

    # Weighted composite: market 40%, tech 30%, risk 30%
    scores = []
    if market:
        scores.append(("market", market.score, 0.4))
    if tech:
        scores.append(("tech", tech.score, 0.3))
    if risk:
        scores.append(("risk", risk.score, 0.3))

    if scores:
        total_weight = sum(w for _, _, w in scores)
        composite = sum(s * w for _, s, w in scores) / total_weight
    else:
        composite = 0.0

    if composite >= 7:
        recommendation = "Invest"
    elif composite >= 5:
        recommendation = "Needs More Research"
    else:
        recommendation = "Pass"

    return InvestmentReport(
        business_idea=business_idea,
        market=market,
        tech=tech,
        risk=risk,
        composite_score=round(composite, 1),
        recommendation=recommendation,
    )


# Run Parallel again with the custom merge
parallel_with_merge = Parallel(
    steps=[market_analyst, tech_analyst, risk_analyst],
    merge=merge_into_report,
    name="investment_analysis_merged",
)

merged_result = await parallel_with_merge.run(business_idea)
report: InvestmentReport = merged_result.merged

print(f"Composite score: {report.composite_score}/10")
print(f"Recommendation:  {report.recommendation}")
print(f"\nBreakdown:")
if report.market:
    print(f"  Market:    {report.market.score}/10")
if report.tech:
    print(f"  Technical: {report.tech.score}/10")
if report.risk:
    print(f"  Risk:      {report.risk.score}/10")

Composite score: 6.8/10
Recommendation:  Needs More Research

Breakdown:
  Market:    8/10
  Technical: 8/10
  Risk:      4/10


## 5. Timeout demo — graceful degradation

What happens when one step is too slow? `Parallel` accepts a `timeout` parameter (in seconds). Steps that exceed it are cancelled and recorded as `TimeoutError` in `result.errors`. The merge function still runs on whatever succeeded.

We add a deliberately slow 4th agent (a `FunctionStep` with a sleep) to demonstrate this.

In [6]:
import asyncio
from orqest.orchestration import FunctionStep


async def slow_regulatory_check(input_data: str) -> str:
    """Simulates a very slow regulatory compliance check."""
    await asyncio.sleep(60)  # Will never finish before our timeout
    return "Regulatory check complete"


# Parallel with 3 real analysts + 1 slow step, 30s timeout
parallel_with_timeout = Parallel(
    steps=[market_analyst, tech_analyst, risk_analyst, slow_regulatory_check],
    merge=merge_into_report,
    timeout=30.0,
    name="investment_analysis_timeout",
)

t0 = time.monotonic()
timeout_result = await parallel_with_timeout.run(business_idea)
timeout_elapsed = time.monotonic() - t0

print(f"Completed in {timeout_elapsed:.2f}s (with 30s timeout)")
print(f"Successful outputs: {sum(1 for o in timeout_result.outputs if o is not None)}/4")
print(f"Errors: {sum(1 for e in timeout_result.errors if e is not None)}/4")

# Show which steps failed
for i, error in enumerate(timeout_result.errors):
    if error is not None:
        print(f"\n  Step {i} error: {type(error).__name__}: {error}")

# The merge still produced a valid report from the 3 successful analysts
timeout_report: InvestmentReport = timeout_result.merged
print(f"\nReport still generated despite timeout!")
print(f"Composite score: {timeout_report.composite_score}/10")
print(f"Recommendation:  {timeout_report.recommendation}")

Completed in 30.03s (with 30s timeout)
Successful outputs: 3/4
Errors: 1/4

  Step 3 error: TimeoutError: Step 'slow_regulatory_check' timed out

Report still generated despite timeout!
Composite score: 6.8/10
Recommendation:  Needs More Research


## What's happening under the hood (Parallel)

1. `Parallel.__init__()` calls `_coerce_step()` on each item — `BaseAgent` instances become `AgentStep`, async callables become `FunctionStep`
2. `parallel.run(input_data)` creates one `asyncio.Task` per step via `asyncio.create_task(step.execute(input_data))`
3. `asyncio.wait(tasks, timeout=...)` waits for all tasks concurrently; if a timeout is set, tasks still running after that deadline land in the `pending` set
4. Pending tasks are cancelled and recorded as `TimeoutError` in `result.errors`
5. Completed tasks are checked: exceptions go into `errors`, successful values go into `outputs` — both lists are indexed by step position
6. The merge function receives only the successful (non-None) outputs and produces the `merged` value
7. `ParallelResult` is a frozen dataclass with `.outputs`, `.errors`, and `.merged`

---

# Part 2: Router — Customer Support Triage

A `Router` selects **one** route based on input and executes only that step. Two selection modes:

1. **Rule-based** — each `Route` has a `condition` callable; first match wins
2. **Classifier-based** — an async function (or agent) inspects the input and returns the route name

We build a customer support system with three specialists:
- **BillingAgent** — handles payment and subscription questions
- **TechSupportAgent** — handles bugs, errors, and technical issues
- **GeneralAgent** — handles everything else (fallback)

## 6. Define support agents

Three specialists, each with a focused system prompt. They all share the same output type since a customer support reply has a consistent shape regardless of department.

In [7]:
class SupportReply(BaseModel):
    """Structured customer support response."""
    department: str = Field(description="Which department handled this: billing, technical, or general")
    reply: str = Field(description="The support reply to send to the customer")
    follow_up_needed: bool = Field(description="Whether this needs escalation or follow-up")


class BillingAgent(BaseAgent[GlobalState, SupportReply]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="billing_agent",
            system_prompt=(
                "You are a billing support specialist. You handle questions about "
                "payments, invoices, subscriptions, refunds, and pricing. Be helpful "
                "and specific. Always set department to 'billing'."
            ),
            output_type=SupportReply,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> SupportReply:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


class TechSupportAgent(BaseAgent[GlobalState, SupportReply]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="tech_support_agent",
            system_prompt=(
                "You are a technical support specialist. You handle bug reports, "
                "error messages, configuration issues, and technical troubleshooting. "
                "Ask clarifying questions when needed. Always set department to 'technical'."
            ),
            output_type=SupportReply,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> SupportReply:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


class GeneralAgent(BaseAgent[GlobalState, SupportReply]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="general_agent",
            system_prompt=(
                "You are a general customer support agent. You handle questions that "
                "don't fit into billing or technical categories — product information, "
                "account settings, feature requests, and general inquiries. "
                "Always set department to 'general'."
            ),
            output_type=SupportReply,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> SupportReply:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


billing_agent = BillingAgent(model=config.llm_model, api_key=config.llm_api_key)
tech_support_agent = TechSupportAgent(model=config.llm_model, api_key=config.llm_api_key)
general_agent = GeneralAgent(model=config.llm_model, api_key=config.llm_api_key)

print("Support agents defined: BillingAgent, TechSupportAgent, GeneralAgent")

Support agents defined: BillingAgent, TechSupportAgent, GeneralAgent


## 7. Rule-based router — keyword matching

Each `Route` has a `condition` callable that receives the input and returns `True`/`False`. Routes are evaluated in order — first match wins. This is fast and deterministic (no LLM call for routing).

In [8]:
from orqest.orchestration import Route, Router

BILLING_KEYWORDS = {"bill", "invoice", "payment", "subscription", "refund", "charge", "pricing", "plan"}
TECH_KEYWORDS = {"error", "bug", "crash", "broken", "not working", "500", "timeout", "exception", "debug"}


def is_billing(input_data: str) -> bool:
    """Check if the message contains billing-related keywords."""
    words = input_data.lower().split()
    return bool(BILLING_KEYWORDS & set(words))


def is_technical(input_data: str) -> bool:
    """Check if the message contains technical keywords."""
    lower = input_data.lower()
    return any(kw in lower for kw in TECH_KEYWORDS)


rule_router = Router(
    routes=[
        Route("billing", billing_agent, condition=is_billing),
        Route("technical", tech_support_agent, condition=is_technical),
    ],
    fallback=general_agent,
    name="support_router",
)

# Test with a billing question
billing_message = "I was charged twice on my last invoice. Can I get a refund for the duplicate payment?"

print(f"Message: {billing_message[:60]}...")
reply = await rule_router.run(billing_message)
print(f"Routed to: {reply.department}")
print(f"Reply: {reply.reply[:200]}...")
print(f"Follow-up needed: {reply.follow_up_needed}")

Message: I was charged twice on my last invoice. Can I get a refund f...


Routed to: billing
Reply: I'm sorry to hear that you were charged twice on your last invoice. I will review your account for duplicate payments and initiate a refund for the extra charge if confirmed. Can you please provide yo...
Follow-up needed: True


In [9]:
# Test with a technical issue
tech_message = "I keep getting a 500 error when I try to upload files larger than 10MB. The crash happens every time."

print(f"Message: {tech_message[:60]}...")
reply = await rule_router.run(tech_message)
print(f"Routed to: {reply.department}")
print(f"Reply: {reply.reply[:200]}...")
print(f"Follow-up needed: {reply.follow_up_needed}")

Message: I keep getting a 500 error when I try to upload files larger...


Routed to: technical
Reply: A 500 error typically indicates a server-side issue, and the consistent crashes with files larger than 10MB suggest there might be a file size limit set in your server or application configuration. To...
Follow-up needed: True


## 8. Fallback behavior

When no route condition matches, the `fallback` step handles the request. If no fallback is configured, the router raises `RouterError`.

In [10]:
# This message doesn't match billing or technical keywords — falls back to GeneralAgent
general_message = "What are your office hours? I'd like to visit your headquarters."

print(f"Message: {general_message[:60]}...")
reply = await rule_router.run(general_message)
print(f"Routed to: {reply.department}")
print(f"Reply: {reply.reply[:200]}...")
print(f"Follow-up needed: {reply.follow_up_needed}")

Message: What are your office hours? I'd like to visit your headquart...


Routed to: general
Reply: Thank you for your interest in visiting our headquarters! Our standard office hours are Monday through Friday, from 9:00 AM to 5:00 PM. If you would like to schedule a visit or need specific instructi...
Follow-up needed: False


In [11]:
from orqest.orchestration import RouterError

# Without a fallback, unmatched messages raise RouterError
router_no_fallback = Router(
    routes=[
        Route("billing", billing_agent, condition=is_billing),
        Route("technical", tech_support_agent, condition=is_technical),
    ],
    name="strict_router",
)

try:
    await router_no_fallback.run("What are your office hours?")
except RouterError as e:
    print(f"RouterError caught: {e}")

RouterError caught: No route matched for input and no fallback configured (router='strict_router').


## 9. Classifier-based router — async function classifier

Instead of keyword rules, you can pass a `classifier` — an async function that inspects the input and returns the route name as a string. This is more flexible and can handle ambiguous messages that keyword matching would miss.

Here we write a simple keyword classifier as an async function. In production, you could replace this with an LLM-based classifier agent.

In [12]:
async def classify_support_message(message: str) -> str:
    """Classify a customer message into a route name using keyword scoring.

    Returns one of: 'billing', 'technical', 'general'.
    More sophisticated than simple keyword matching — scores each category
    and picks the highest.
    """
    lower = message.lower()

    billing_score = sum(1 for kw in BILLING_KEYWORDS if kw in lower)
    tech_score = sum(1 for kw in TECH_KEYWORDS if kw in lower)

    # Additional contextual signals
    if any(phrase in lower for phrase in ["how much", "cost", "price", "upgrade", "downgrade"]):
        billing_score += 2
    if any(phrase in lower for phrase in ["doesn't work", "help me fix", "stack trace", "log"]):
        tech_score += 2

    if billing_score > tech_score and billing_score > 0:
        return "billing"
    elif tech_score > billing_score and tech_score > 0:
        return "technical"
    return "general"


# Create router with the classifier — no conditions needed on routes
classifier_router = Router(
    routes=[
        Route("billing", billing_agent),
        Route("technical", tech_support_agent),
        Route("general", general_agent),
    ],
    classifier=classify_support_message,
    name="classifier_router",
)

# Test all 3 message types
test_messages = [
    ("How much does the premium plan cost? I want to upgrade.", "billing"),
    ("The app doesn't work after the latest update. Help me fix this.", "technical"),
    ("Can you tell me more about your company's sustainability initiatives?", "general"),
]

for message, expected in test_messages:
    reply = await classifier_router.run(message)
    status = "correct" if reply.department == expected else "MISMATCH"
    print(f"[{status}] Expected: {expected:10s} | Got: {reply.department:10s} | {message[:50]}...")

[correct] Expected: billing    | Got: billing    | How much does the premium plan cost? I want to upg...


[correct] Expected: technical  | Got: technical  | The app doesn't work after the latest update. Help...


[correct] Expected: general    | Got: general    | Can you tell me more about your company's sustaina...


## What's happening under the hood (Router)

1. `Router.__init__()` validates that either route conditions or a classifier exist, then builds a `_route_map` dict keyed by route name
2. `router.run(input_data)` calls `_select_route(input_data)` to pick a route name
3. **Rule-based mode**: iterates routes in order, calls each `condition(input_data)`, returns the first `True` match
4. **Classifier mode**: calls the classifier with `input_data` — if it is a `BaseAgent`, a fresh `GlobalState` is created and the agent is run; if it is an async callable, it is called directly
5. If a route name is found in `_route_map`, its step is executed via `step.execute(input_data)`
6. If no route matches and a `fallback` is configured, the fallback step handles the request
7. If no route matches and no fallback exists, `RouterError` is raised
8. Each route's step is coerced via `_coerce_step()` — same as `Parallel`

## 10. Assessment — timing comparison & routing accuracy

Let's compare parallel vs sequential execution time to quantify the concurrency benefit, and summarize the routing results.

In [13]:
# --- Sequential baseline for comparison ---

def _make_state(message: str) -> GlobalState:
    """Create a fresh GlobalState with a user message."""
    state = GlobalState()
    state.add_message("user", message)
    return state


t0 = time.monotonic()
seq_market = await market_analyst.run(_make_state(business_idea))
seq_tech = await tech_analyst.run(_make_state(business_idea))
seq_risk = await risk_analyst.run(_make_state(business_idea))
sequential_elapsed = time.monotonic() - t0

print("=== Timing Comparison ===")
print(f"Parallel execution:   {parallel_elapsed:.2f}s")
print(f"Sequential execution: {sequential_elapsed:.2f}s")
speedup = sequential_elapsed / parallel_elapsed if parallel_elapsed > 0 else 0
print(f"Speedup:              {speedup:.1f}x faster with Parallel")
print()
print("=== Routing Summary ===")
print(f"Rule-based router:      3 test cases (billing, technical, fallback)")
print(f"Classifier-based router: 3 test cases (billing, technical, general)")
print(f"Fallback behavior:      RouterError raised when no fallback configured")
print()
print("=== Key Takeaways ===")
print("- Parallel runs N agents in ~1x time instead of ~Nx time")
print("- ParallelResult gives you per-step outputs AND errors — no silent failures")
print("- Custom merge functions let you combine heterogeneous outputs")
print("- Timeout keeps the system responsive even when one step hangs")
print("- Router conditions are evaluated in order — put specific routes first")
print("- Classifier-based routing is more flexible for ambiguous inputs")

=== Timing Comparison ===
Parallel execution:   4.04s
Sequential execution: 8.48s
Speedup:              2.1x faster with Parallel

=== Routing Summary ===
Rule-based router:      3 test cases (billing, technical, fallback)
Classifier-based router: 3 test cases (billing, technical, general)
Fallback behavior:      RouterError raised when no fallback configured

=== Key Takeaways ===
- Parallel runs N agents in ~1x time instead of ~Nx time
- ParallelResult gives you per-step outputs AND errors — no silent failures
- Custom merge functions let you combine heterogeneous outputs
- Timeout keeps the system responsive even when one step hangs
- Router conditions are evaluated in order — put specific routes first
- Classifier-based routing is more flexible for ambiguous inputs
